Phase 4 — Databricks

/Créer un cluster et un notebook

/ Monter le Data Lake

/Configurer le mount vers votre container ADLS.
/Vérifier le montage

In [0]:
storage_account_name = "mbdatalake"
container_name = "container"
storage_account_key = "********************************"

# Au lieu de configurer globalement, on va essayer de lire directement en passant les options à Spark
path_bronze = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"

print("Tentative de lecture directe avec options...")

# On essaie de charger un DataFrame vide juste pour tester la connexion au chemin
try:
    df_test = spark.read \
        .format("parquet") \
        .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key) \
        .load(path_bronze + "/product") # On teste sur le dossier product
    
    print("Connexion réussie ! Spark arrive à voir le dossier product.")
    display(df_test.limit(5))
except Exception as e:
    print("Erreur de lecture directe :", e)

Tentative de lecture directe avec options...
Connexion réussie ! Spark arrive à voir le dossier product.
Erreur de lecture directe : [PATH_NOT_FOUND] Path does not exist: abfss://container@mbdatalake.dfs.core.windows.net/bronze/product. SQLSTATE: 42K03

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at org.apache.spark.sql.errors.QueryCompilationErrors$.dataPathNotExistError(QueryCompilationErrors.scala:3018)
	at org.apache.spark.sql.execution.datasources.DataSource$.$anonfun$checkAndGlobPathIfNecessary$1(DataSource.scala:1188)
	at scala.collection.immutable.List.flatMap(List.scala:294)
	at scala.collection.immutable.List.flatMap(List.scala:79)
	at org.apache.spark.sql.execution.datasources.DataSource$.checkAndGlobPathIfNecessary(DataSource.scala:1169)
	at org.apache.spark.sql.execution.datasources.DataSource.checkAndGlobPathIfNecessary(DataSource.scala:778)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:599)
	at org.apache.spark.sql

In [0]:
storage_account_name = "mbdatalake"
container_name = "container"
storage_account_key = "********************************"

# Base du chemin d'accès
base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"

# Dictionnaire de configuration de sécurité pour la lecture directe
azure_options = {
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net": storage_account_key
}

print("Chargement des données Bronze en cours...")

try:
    # 1. Chargement des Produits
    df_products = spark.read.format("parquet").options(**azure_options).load(f"{base_path}/product")
    print("-> Table 'product' chargée avec succès.")
    
    # 2. Chargement des Magasins
    df_stores = spark.read.format("parquet").options(**azure_options).load(f"{base_path}/store")
    print("-> Table 'store' chargée avec succès.")
    
    # 3. Chargement des Transactions
    df_transactions = spark.read.format("parquet").options(**azure_options).load(f"{base_path}/transaction")
    print("-> Table 'transaction' chargée avec succès.")
    
    # 4. Chargement des Clients (API JSON convertie en Parquet par ADF)
    df_customers = spark.read.format("parquet").options(**azure_options).load(f"{base_path}/customer")
    print("-> Table 'customer' chargée avec succès.")
    
    print("\nFélicitations ! Toutes les sources de la couche Bronze sont montées en mémoire.")

except Exception as e:
    print("\nErreur lors du chargement d'une table :")
    print(e)

Chargement des données Bronze en cours...
-> Table 'product' chargée avec succès.
-> Table 'store' chargée avec succès.
-> Table 'transaction' chargée avec succès.
-> Table 'customer' chargée avec succès.

Félicitations ! Toutes les sources de la couche Bronze sont montées en mémoire.


In [0]:
storage_account_name = "mbdatalake"
container_name = "container"
storage_account_key = "********************************"

base_url = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/bronze"
azure_options = {
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net": storage_account_key
}

# --- TEST DE LECTURE DU FICHIER PRODUCT ---
print("Vérification du fichier Product...")
try:
    # On pointe sur le nom exact de ta capture : dossier 'product' / fichier 'dbo.products.parquet'
    df_products = spark.read.format("parquet").options(**azure_options).load(f"{base_url}/product/dbo.products.parquet")
    
    # Action immédiate pour forcer Spark Connect à valider le chemin
    df_products.take(1) 
    print("🎉 SUCCÈS ! Le fichier Product est accessible et lu par Spark.")
    
    # Affichage du schéma directement ici
    print("\n--- Schéma de la table PRODUCTS ---")
    df_products.printSchema()

except Exception as e:
    print("❌ ÉCHEC de lecture. L'erreur retournée est :")
    print(e)

In [0]:
storage_account_name = "mbdatalake"
container_name = "container"
storage_account_key = "********************************"

# ATTENTION : Correction de la casse -> 'Bronze' avec un B majuscule
base_url = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Bronze"

azure_options = {
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net": storage_account_key
}

print("Vérification du fichier Product avec le bon chemin (casse corrigée)...")

try:
    # On pointe vers le DOSSIER 'product' (singulier) situé dans 'Bronze' (Majuscule)
    df_products = spark.read.format("parquet").options(**azure_options).load(f"{base_url}/product")
    
    # On force la lecture d'une ligne pour valider
    df_products.take(1)
    print("🎉 Ça fonctionne ")
    
    print("\n--- Schéma de la table PRODUCTS ---")
    df_products.printSchema()

except Exception as e:
    print("❌ Toujours un échec. Erreur :")
    print(e)

Vérification du fichier Product avec le bon chemin (casse corrigée)...
🎉 Ça fonctionne 

--- Schéma de la table PRODUCTS ---
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: decimal(10,2) (nullable = true)



In [0]:
storage_account_name = "mbdatalake"
container_name = "container"
storage_account_key = "********************************"
# Le chemin validé (avec 'Bronze' en majuscule)
base_url = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Bronze"
azure_options = {
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net": storage_account_key
}

print("Chargement global de la couche Bronze...")

try:
    # 1. Products
    df_products = spark.read.format("parquet").options(**azure_options).load(f"{base_url}/product")
    df_products.take(1)
    print("-> df_products chargé.")

    # 2. Customers
    df_customers = spark.read.format("parquet").options(**azure_options).load(f"{base_url}/customer")
    df_customers.take(1)
    print("-> df_customers chargé.")

    # 3. Stores
    df_stores = spark.read.format("parquet").options(**azure_options).load(f"{base_url}/store")
    df_stores.take(1)
    print("-> df_stores chargé.")

    # 4. Transactions
    df_transactions = spark.read.format("parquet").options(**azure_options).load(f"{base_url}/transaction")
    df_transactions.take(1)
    print("-> df_transactions chargé.")

    print("\n--- TOUS LES SCHÉMAS ---")
    print("\n[CUSTOMERS]")
    df_customers.printSchema()
    
    print("\n[STORES]")
    df_stores.printSchema()
    
    print("\n[TRANSACTIONS]")
    df_transactions.printSchema()

except Exception as e:
    print("❌ Erreur sur l'une des tables :", e)

Chargement global de la couche Bronze...
-> df_products chargé.
-> df_customers chargé.
-> df_stores chargé.
-> df_transactions chargé.

--- TOUS LES SCHÉMAS ---

[CUSTOMERS]
root
 |-- customer_id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: string (nullable = true)


[STORES]
root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- location: string (nullable = true)


[TRANSACTIONS]
root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- transaction_date: date (nullable = true)



Phase 5 _ Traitement Silver : Nettoyage et Standardisation des données

In [0]:
from pyspark.sql.functions import col, to_date

print("Début du nettoyage et de la standardisation (Couche Silver)...")

# 1. Nettoyage de CUSTOMERS (Dédoublonnement + Cast de la date)
# On convertit 'registration_date' de String à Date (en supposant le format standard AAAA-MM-JJ)
df_customers_silver = df_customers \
    .dropDuplicates(["customer_id"]) \
    .withColumn("registration_date", to_date(col("registration_date")))

# 2. Nettoyage de PRODUCTS (Dédoublonnement)
df_products_silver = df_products \
    .dropDuplicates(["product_id"])

# 3. Nettoyage de STORES (Dédoublonnement)
df_stores_silver = df_stores \
    .dropDuplicates(["store_id"])

# 4. Nettoyage de TRANSACTIONS (Dédoublonnement + Filtrage des lignes sans ID de transaction)
df_transactions_silver = df_transactions \
    .dropDuplicates(["transaction_id"]) \
    .filter(col("transaction_id").isNotNull())

print("🎉 Données nettoyées avec succès ! Vérification du nouveau schéma des clients :")
df_customers_silver.printSchema()

Début du nettoyage et de la standardisation (Couche Silver)...
🎉 Données nettoyées avec succès ! Vérification du nouveau schéma des clients :
root
 |-- customer_id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- city: string (nullable = true)
 |-- registration_date: date (nullable = true)



Phase 5 _ Traitement Silver : Écriture des données dans la couche Silver

In [0]:
from pyspark.sql.functions import col, round

print("Enrichissement de la couche Silver (Jointures & Calculs)...")

try:
    # 15. Réaliser les jointures 
    # On part des transactions et on récupère le prix du produit associé
    df_transactions_enriched = df_transactions_silver.join(
        df_products_silver.select("product_id", "price", "product_name"),
        on="product_id",
        how="inner"
    )

    # 16. Ajouter un indicateur calculé (total_amount = quantity * price)
    # On arrondit à 2 décimales pour rester propre financièrement
    df_transactions_final = df_transactions_enriched.withColumn(
        "total_amount", 
        round(col("quantity") * col("price"), 2)
    )

    print("-> Jointures réussies et colonne 'total_amount' calculée.")
    
    # 17. Sauvegarder Silver au format Delta
    silver_base_url = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Silver"
    
    print("\nÉcriture des tables finales dans le Data Lake (Format Delta)...")
    
    # Sauvegarde de la table de faits (Transactions enrichies)
    print("-> Sauvegarde de 'transaction' (enrichie)...")
    df_transactions_final.write.format("delta").options(**azure_options).mode("overwrite").save(f"{silver_base_url}/transaction")
    
    # Sauvegarde des tables de dimensions nettoyées
    print("-> Sauvegarde de 'customer'...")
    df_customers_silver.write.format("delta").options(**azure_options).mode("overwrite").save(f"{silver_base_url}/customer")
    
    print("-> Sauvegarde de 'product'...")
    df_products_silver.write.format("delta").options(**azure_options).mode("overwrite").save(f"{silver_base_url}/product")
    
    print("-> Sauvegarde de 'store'...")
    df_stores_silver.write.format("delta").options(**azure_options).mode("overwrite").save(f"{silver_base_url}/store")

    print("\n🎉 [SUCCÈS] Les étapes 15, 16 et 17 du brief sont validées en format Delta ! 🎉")
    
    # Petit aperçu pour valider visuellement le montant calculé
    print("\nAperçu des transactions enrichies :")
    df_transactions_final.select("transaction_id", "product_id", "quantity", "price", "total_amount").show(5)

except Exception as e:
    print("\n❌ Erreur durant l'exécution du brief Silver :")
    print(e)

Enrichissement de la couche Silver (Jointures & Calculs)...
-> Jointures réussies et colonne 'total_amount' calculée.

Écriture des tables finales dans le Data Lake (Format Delta)...
-> Sauvegarde de 'transaction' (enrichie)...
-> Sauvegarde de 'customer'...
-> Sauvegarde de 'product'...
-> Sauvegarde de 'store'...

🎉 [SUCCÈS] Les étapes 15, 16 et 17 du brief sont validées en format Delta ! 🎉

Aperçu des transactions enrichies :
+--------------+----------+--------+------+------------+
|transaction_id|product_id|quantity| price|total_amount|
+--------------+----------+--------+------+------------+
|            12|         5|       4| 35.00|      140.00|
|            18|         1|       3|149.99|      449.97|
|            16|         6|       4| 89.00|      356.00|
|             5|         5|       1| 35.00|       35.00|
|            10|         2|       5|399.50|     1997.50|
+--------------+----------+--------+------+------------+
only showing top 5 rows


Phase 6 — Traitement Gold : Agrégations et KPIs Métiers

In [0]:
from pyspark.sql.functions import sum, count, avg, round

print("Démarrage du traitement de la couche Gold...")

# 18. Lire les données depuis la couche Silver
silver_base_url = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Silver"
gold_base_url = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Gold"

try:
    print("-> Lecture de la table Silver 'transaction'...")
    df_silver_sales = spark.read.format("delta").options(**azure_options).load(f"{silver_base_url}/transaction")

    # 19. Réaliser les agrégations métier (Regroupement par Store et par Produit)
    print("-> Calcul des KPIs métiers (Quantités, CA, Transactions, Panier Moyen)...")
    df_gold_kpis = df_silver_sales.groupBy("store_id", "product_id") \
        .agg(
            sum("quantity").alias("total_quantity_sold"),
            round(sum("total_amount"), 2).alias("total_revenue"),
            count("transaction_id").alias("transaction_count"),
            round(avg("total_amount"), 2).alias("avg_transaction_value")
        )

    # 20. Sauvegarder Gold au format Delta
    print("\n-> Écriture du rapport KPI final dans la couche Gold...")
    df_gold_kpis.write \
        .format("delta") \
        .options(**azure_options) \
        .mode("overwrite") \
        .save(f"{gold_base_url}/kpis_store_product")

    print("🎉 [SUCCÈS] La couche Gold a été générée et sauvegardée ! 🎉")

    # Affichage du résultat final pour vérification
    print("\nAperçu des indicateurs Gold (Top 5) :")
    df_gold_kpis.orderBy("total_revenue", ascending=False).show(5)

except Exception as e:
    print("\n❌ Erreur durant le traitement de la couche Gold :")
    print(e)

Démarrage du traitement de la couche Gold...
-> Lecture de la table Silver 'transaction'...
-> Calcul des KPIs métiers (Quantités, CA, Transactions, Panier Moyen)...

-> Écriture du rapport KPI final dans la couche Gold...
🎉 [SUCCÈS] La couche Gold a été générée et sauvegardée ! 🎉

Aperçu des indicateurs Gold (Top 5) :
+--------+----------+-------------------+-------------+-----------------+---------------------+
|store_id|product_id|total_quantity_sold|total_revenue|transaction_count|avg_transaction_value|
+--------+----------+-------------------+-------------+-----------------+---------------------+
|       5|         7|                  5|      6495.00|                2|              3247.50|
|       3|         7|                  5|      6495.00|                1|              6495.00|
|       2|         7|                  5|      6495.00|                1|              6495.00|
|       2|         2|                  8|      3196.00|                2|              1598.00|
|      

Étape 7 : Exporter la table Gold en un seul fichier CSV

In [0]:
from pyspark.sql.functions import sum, count, avg, round

# --- CONFIGURATION STRICTE DES ACCÈS AZURE ---
storage_account_name = "mbdatalake"
container_name = "container"
storage_account_key = "********************************"

azure_options = {
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net": storage_account_key
}

# --- CHEMINS DE L'ARCHITECTURE ---
silver_base_url = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Silver"
gold_csv_url = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Gold/export_sales_kpis"

print("1. Lecture de la couche Silver...")
try:
    # Récupération de la table de faits Silver
    df_silver_sales = spark.read.format("delta").options(**azure_options).load(f"{silver_base_url}/transaction")

    print("2. Calcul des KPIs métiers (Génération de la table Gold)...")
    # Génération de la table Gold
    df_gold_kpis = df_silver_sales.groupBy("store_id", "product_id") \
        .agg(
            sum("quantity").alias("total_quantity_sold"),
            round(sum("total_amount"), 2).alias("total_revenue"),
            count("transaction_id").alias("transaction_count"),
            round(avg("total_amount"), 2).alias("avg_transaction_value")
        )

    print("3. Exportation immédiate en format CSV unique...")
    # coalesce(1) force Spark à tout compiler dans un seul fichier CSV propre
    df_gold_kpis.coalesce(1).write \
        .format("csv") \
        .options(**azure_options) \
        .option("header", "true") \
        .option("delimiter", ",") \
        .mode("overwrite") \
        .save(gold_csv_url)

    print("\n🎉 [SUCCÈS GLOBAL] Le pipeline est arrivé à son terme !")
    print(f"Ton unique fichier CSV est généré dans le dossier : Gold/export_sales_kpis")

except Exception as e:
    print("\n❌ Erreur technique rencontrée :")
    print(e)

1. Lecture de la couche Silver...
2. Calcul des KPIs métiers (Génération de la table Gold)...
3. Exportation immédiate en format CSV unique...

🎉 [SUCCÈS GLOBAL] Le pipeline est arrivé à son terme !
Ton unique fichier CSV est généré dans le dossier : Gold/export_sales_kpis


In [0]:
from pyspark.sql.functions import col

# --- CONFIGURATION STRICTE DES ACCÈS AZURE ---
storage_account_name = "mbdatalake"
container_name = "container"
storage_account_key = "********************************"

azure_options = {
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net": storage_account_key
}

# --- CHEMINS SOURCE ET DESTINATION ---
silver_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Silver"
gold_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/Gold"

print("🚀 Démarrage de la conversion globale Silver Delta -> Gold CSV...")

try:
    # 1. Traitement de la table PRODUCT
    print("-> Rechargement et export de 'product'...")
    df_prod = spark.read.format("delta").options(**azure_options).load(f"{silver_base_path}/product")
    df_prod.coalesce(1).write.format("csv").options(**azure_options).option("header", "true").mode("overwrite").save(f"{gold_base_path}/export_product_csv")

    # 2. Traitement de la table STORE
    print("-> Rechargement et export de 'store'...")
    df_sto = spark.read.format("delta").options(**azure_options).load(f"{silver_base_path}/store")
    df_sto.coalesce(1).write.format("csv").options(**azure_options).option("header", "true").mode("overwrite").save(f"{gold_base_path}/export_store_csv")

    # 3. Traitement de la table CUSTOMER
    print("-> Rechargement et export de 'customer'...")
    df_cust = spark.read.format("delta").options(**azure_options).load(f"{silver_base_path}/customer")
    df_cust.coalesce(1).write.format("csv").options(**azure_options).option("header", "true").mode("overwrite").save(f"{gold_base_path}/export_customer_csv")

    # 4. Traitement de la table TRANSACTION (Niveau fin pour l'axe temporel Date)
    print("-> Rechargement et export de 'transaction'...")
    df_trans = spark.read.format("delta").options(**azure_options).load(f"{silver_base_path}/transaction")
    df_trans.coalesce(1).write.format("csv").options(**azure_options).option("header", "true").mode("overwrite").save(f"{gold_base_path}/export_transaction_csv")

    print("\n🎉 [SUCCÈS TOTAL] Toutes tes dimensions et faits sont convertis en CSV dans le dossier Gold !")
    print("Tu peux rafraîchir ton portail Azure, tout t'attend sagement dans le conteneur 'Gold'.")

except Exception as e:
    print("\n❌ Erreur lors de l'exécution du script :")
    print(e)

🚀 Démarrage de la conversion globale Silver Delta -> Gold CSV...
-> Rechargement et export de 'product'...
-> Rechargement et export de 'store'...
-> Rechargement et export de 'customer'...
-> Rechargement et export de 'transaction'...

🎉 [SUCCÈS TOTAL] Toutes tes dimensions et faits sont convertis en CSV dans le dossier Gold !
Tu peux rafraîchir ton portail Azure, tout t'attend sagement dans le conteneur 'Gold'.
